In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

"""# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session"""

'# Input data files are available in the read-only "../input/" directory\n# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory\n\nimport os\nfor dirname, _, filenames in os.walk(\'/kaggle/input\'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n\n# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" \n# You can also write temporary files to /kaggle/temp/, but they won\'t be saved outside of the current session'

In [2]:
train_hr="/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_High_Resolution"
train_lr="/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution"
test_lr="/kaggle/input/competitions/plant-leaves-super-resolution-challenge/test_Low_Resolution"

In [3]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [4]:
import os
import glob
import random
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset
from PIL import Image

class CropsDataset(Dataset):
    def __init__(self,lr_dir,hr_dir=None):
        self.lr_paths=sorted(glob.glob(os.path.join(lr_dir, "*.png")))
        self.hr_dir=hr_dir

        self.lr_transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])])
        self.hr_transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])])
                                              
    def __len__(self):
        return len(self.lr_paths)

    def __getitem__(self,idx):
        lr_img=Image.open(self.lr_paths[idx]).convert('RGB')

        if self.hr_dir:
            file=os.path.basename(self.lr_paths[idx])
            hr_path=os.path.join(self.hr_dir,file)
            hr_img=Image.open(hr_path).convert('RGB')
            if random.random()>0.5:
                hr_img=TF.hflip(hr_img)
                lr_img=TF.hflip(lr_img)
            if random.random()>0.5:
                hr_img=TF.vflip(hr_img)
                lr_img=TF.vflip(lr_img)
            k=random.choice([0,1,2,3])
            if k > 0:
                lr_img = TF.rotate(lr_img, 90*k)
                hr_img = TF.rotate(hr_img, 90*k)
            hr=self.hr_transform(hr_img)
            lr=self.lr_transform(lr_img)
            return lr,hr
            
        lr=self.lr_transform(lr_img)
        return lr,os.path.basename(self.lr_paths[idx])

data=CropsDataset(train_lr,train_hr)
lr_sample,hr_sample=data[0]
print(lr_sample.shape)
print(hr_sample.shape)
len(data)

torch.Size([3, 32, 32])
torch.Size([3, 128, 128])


1642

In [5]:
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(channels,channels,3,1,1),
            nn.PReLU(),
            nn.Conv2d(channels,channels,3,1,1)
        )
    def forward(self,x):
            return x+0.2*self.block(x)

In [6]:
class Generator(nn.Module):
    def __init__(self,num_blocks=16):
        super().__init__()
        self.head=nn.Conv2d(3,64,3,1,1)
        self.body=nn.Sequential(*[ResidualBlock(64) for _ in range(num_blocks)])
        self.body_out=nn.Conv2d(64,64,3,1,1)

        self.upsample=nn.Sequential(
            nn.Conv2d(64, 256, 3, 1, 1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(64, 256, 3, 1, 1),
            nn.PixelShuffle(2),
            nn.PReLU(),
        )
        self.tail=nn.Sequential(
            nn.Conv2d(64,64,3,1,1),
            nn.PReLU(),
            nn.Conv2d(64,3,3,1,1),
            nn.Tanh()
        )
        
    def forward(self,x):
        head=self.head(x)
        body=self.body_out(self.body(head))
        net=head+body
        upsample=self.upsample(net)
        return self.tail(upsample)
        

In [7]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.body=nn.Sequential(
            nn.Conv2d(6,32,4,2,1),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Conv2d(32,64,4,2,1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 1, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 1, 4, 1, 1))
        
    def forward(self, x,y):
        final=torch.cat([x,y], dim=1)
        return self.body(final)

In [8]:
import torchvision.models as vgg_module

class PerceptualLoss(nn.Module):
    def __init__(self, vgg_path):
        super().__init__()
        vgg = vgg_module.vgg19()
        vgg.load_state_dict(torch.load(vgg_path, map_location=device))
        self.features = nn.Sequential(
            *list(vgg.features)[:18]
        ).eval()
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, pred, target):
        return nn.functional.l1_loss(self.features(pred),self.features(target))

In [9]:
import torch.optim as optim
import torch
from torch.utils.data import DataLoader

device=("cuda" if torch.cuda.is_available() else "cpu")
G=Generator(num_blocks=23).to(device)
D=Discriminator().to(device)
vgg_path="/kaggle/input/competitions/plant-leaves-super-resolution-challenge/vgg19_weights.pth"
perceptual_loss=PerceptualLoss(vgg_path).to(device)

opt_G=optim.Adam(G.parameters(), lr=5e-4, betas=(0.9, 0.999))
opt_D=optim.Adam(D.parameters(), lr=1e-6, betas=(0.9, 0.999))

scheduler_G = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_G, mode='min', factor=0.5, patience=10)
scheduler_D = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_D, mode='min', factor=0.5, patience=10)

criterion_adv=nn.BCEWithLogitsLoss()
criterion_pix=nn.L1Loss()

In [10]:
from torch.utils.data import random_split

full_dataset = CropsDataset(train_lr, train_hr)

val_size = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size],
                                           generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader=DataLoader(CropsDataset(test_lr),batch_size=16,shuffle=False,num_workers=2)

In [11]:
from torch.amp import autocast, GradScaler

scaler_G = GradScaler()
scaler_D = GradScaler()

best_mae = float('inf')
loss_D = torch.tensor(0.5)

for epoch in range(150):
    G.train()
    for lr_imgs, hr_imgs in train_loader:
        lr_imgs = lr_imgs.to(device)
        hr_imgs = hr_imgs.to(device)
        lr_up = torch.nn.functional.interpolate(lr_imgs, scale_factor=4, mode='bicubic', align_corners=False)

        if epoch>20:
            with autocast(device_type='cuda'):
                fake_hr = G(lr_imgs)
                real_pred = D(hr_imgs, lr_up)
                fake_pred = D(fake_hr.detach(), lr_up)
                loss_real = criterion_adv(real_pred, torch.ones_like(real_pred))
                loss_fake = criterion_adv(fake_pred, torch.zeros_like(fake_pred))
                loss_D = (loss_real + loss_fake) * 0.5
    
            opt_D.zero_grad()
            scaler_D.scale(loss_D).backward()
            scaler_D.step(opt_D)
            scaler_D.update()

        with autocast(device_type='cuda'):
            fake_hr = G(lr_imgs)
            fake_pred2 = D(fake_hr, lr_up)
            per_loss = perceptual_loss(fake_hr, hr_imgs)
            loss_adv = criterion_adv(fake_pred2, torch.ones_like(fake_pred2))
            loss_pix = criterion_pix(fake_hr, hr_imgs)
            if epoch<=20:
                loss_G=loss_pix*10+0.009*per_loss
            else:
                adv_weight = 0.005 if loss_D.item() > 0.35 else 0.0
                loss_G = loss_pix * 10 + 0.1 * per_loss + adv_weight * loss_adv 

        opt_G.zero_grad()
        scaler_G.scale(loss_G).backward()
        scaler_G.step(opt_G)
        scaler_G.update()

    G.eval()
    with torch.no_grad():
        val_mae, n = 0.0, 0
        for lr_imgs, hr_imgs in val_loader:
            with autocast(device_type='cuda'):
                fake_hr = G(lr_imgs.to(device))
            val_mae += (torch.mean(torch.abs(fake_hr - hr_imgs.to(device))) * 0.5 * 255).item()
            n += 1
        val_mae /= n

    scheduler_G.step(val_mae)
    scheduler_D.step(val_mae)

    if val_mae < best_mae:
        best_mae = val_mae
        print(f"Saving best model at MAE: {best_mae:.4f}")
        torch.save(G.state_dict(), "generator_best.pth")

    if epoch % 5 == 0:
        print(f"Epoch {epoch} | D: {loss_D.item():.4f} | G: {loss_G.item():.4f} | val_MAE: {val_mae:.4f}")

print("Training complete")

Saving best model at MAE: 17.7540
Epoch 0 | D: 0.5000 | G: 1.3001 | val_MAE: 17.7540
Saving best model at MAE: 17.3109
Saving best model at MAE: 17.1940
Saving best model at MAE: 17.0108
Epoch 5 | D: 0.5000 | G: 1.4136 | val_MAE: 17.0360
Saving best model at MAE: 16.9519
Saving best model at MAE: 16.8676
Epoch 10 | D: 0.5000 | G: 1.2468 | val_MAE: 16.8720
Saving best model at MAE: 16.7996
Saving best model at MAE: 16.7730
Saving best model at MAE: 16.7515
Epoch 15 | D: 0.5000 | G: 1.1846 | val_MAE: 16.7980
Saving best model at MAE: 16.7305
Saving best model at MAE: 16.7046
Saving best model at MAE: 16.6906
Saving best model at MAE: 16.6821
Epoch 20 | D: 0.5000 | G: 1.8668 | val_MAE: 16.6821
Saving best model at MAE: 16.6633
Saving best model at MAE: 16.6516
Saving best model at MAE: 16.6318
Epoch 25 | D: 0.6893 | G: 1.4682 | val_MAE: 16.6898
Saving best model at MAE: 16.6158
Epoch 30 | D: 0.6633 | G: 1.1130 | val_MAE: 16.6158
Epoch 35 | D: 0.6230 | G: 1.1639 | val_MAE: 16.7516
Saving b

In [12]:
device = ("cuda" if torch.cuda.is_available() else "cpu")
G = Generator(num_blocks=23).to(device)
G.load_state_dict(torch.load("/kaggle/working/generator_best.pth", map_location=device))
G.eval()

results = []
with torch.no_grad():
    for lr_imgs, filenames in test_loader:
        lr_imgs = lr_imgs.to(device)

        preds = []
        for k in range(4):
            rotated = torch.rot90(lr_imgs, k, dims=[2, 3])
            sr = G(rotated)
            preds.append(torch.rot90(sr, -k, dims=[2, 3]))

            flipped = torch.flip(rotated, [3])
            sr_f = G(flipped)
            preds.append(torch.rot90(torch.flip(sr_f, [3]), -k, dims=[2, 3]))

        sr_imgs = torch.stack(preds).mean(0)
        sr_imgs = (sr_imgs * 0.5 + 0.5).clamp(0, 1)
        sr_imgs = (sr_imgs * 255).byte()

        for img, fname in zip(sr_imgs, filenames):
            img_np = img.permute(1, 2, 0).cpu().numpy().flatten()
            pixels = " ".join(map(str, img_np))
            results.append({"Id": fname, "Pixels": pixels})

submission = pd.DataFrame(results)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Done!", submission.head())

Done!                          Id                                             Pixels
0  agrivision_test_0000.png  153 146 148 155 147 151 154 145 150 153 144 15...
1  agrivision_test_0001.png  36 34 31 28 26 24 25 23 21 23 21 19 22 19 17 2...
2  agrivision_test_0002.png  143 138 135 144 136 137 145 137 138 147 138 14...
3  agrivision_test_0003.png  162 154 155 163 154 158 163 152 157 163 152 15...
4  agrivision_test_0004.png  117 113 109 116 110 107 117 110 108 116 108 10...
